# NumPy İleri Düzey

Bu notebook, "NumPy Temelleri" ve "NumPy Orta Düzey" notebook'larının devamıdır.
**İleri düzey** konuları yine satır satır Türkçe açıklamalarla ele alır.

İçerik:
1. Bellek Düzeni: strides, C-order ve F-order
2. Gelişmiş Broadcasting Kuralları ve `np.newaxis`
3. Sliding Window (Kayan Pencere) İşlemleri
4. `einsum` ile Genel Tensör İşlemleri
5. Maskeli Array'ler (`np.ma`)
6. Modern Rastgelelik: `Generator` API (`default_rng`)
7. Hızlı Fourier Dönüşümü (FFT)
8. Polinom Uydurma (Curve Fitting)
9. Tarih/Zaman Verisiyle Çalışma (`datetime64`, `timedelta64`)
10. Kısmi Sıralama: `argpartition` ile Performans Kazanımı
11. `apply_along_axis` ile Eksen Bazlı Özel Fonksiyonlar
12. `memmap` ile Bellek Sığmayan Büyük Dosyalarla Çalışma


In [1]:
import numpy as np

# Bu notebook boyunca tekrarlanabilirlik için seed sabitliyoruz
np.random.seed(0)

## 1. Bellek Düzeni: strides, C-order ve F-order

Bir NumPy array'i aslında bellekte **tek boyutlu, ardışık** bir blok olarak durur.
`shape` bize "mantıksal" görünümü verirken, `strides` her eksende bir adım ilerlemek için
bellekte kaç byte atlanması gerektiğini söyler. Bu bilgi, reshape/transpose gibi işlemlerin
neden bu kadar "ucuz" (kopyasız) olduğunu anlamamızı sağlar.

In [2]:
a = np.arange(12).reshape(3, 4)   # 3x4 boyutunda, satır bazlı (C-order, varsayılan) bir array
print("Array:\n", a)
print("shape  :", a.shape)     # (3, 4)
print("strides:", a.strides)   # (32, 8) -> bir sonraki satıra geçmek için 32 byte, sütuna geçmek için 8 byte atlanır
# (8 byte, int64'ün boyutu; bir satırda 4 eleman olduğu için satır atlamak 4*8=32 byte sürer)

Array:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
shape  : (3, 4)
strides: (32, 8)


In [3]:
# transpose (T) işlemi VERİYİ TAŞIMAZ, sadece strides sırasını değiştirir -> çok hızlıdır
a = np.arange(12).reshape(3, 4)
b = a.T   # transpoze alıyoruz

print("Transpoze şekli  :", b.shape)     # (4, 3)
print("Transpoze strides:", b.strides)   # (8, 32) -> strides sırası tersine döndü, veri kopyalanmadı
print("b, a ile bellek paylaşıyor mu?:", b.base is a)  # True -> hâlâ aynı bellek

Transpoze şekli  : (4, 3)
Transpoze strides: (8, 32)
b, a ile bellek paylaşıyor mu?: False


In [4]:
# C-order (satır öncelikli) ve F-order (sütun öncelikli, Fortran tarzı) karşılaştırması
c_array = np.array([[1, 2, 3], [4, 5, 6]], order='C')  # bellekte satır satır dizilir: 1,2,3,4,5,6
f_array = np.array([[1, 2, 3], [4, 5, 6]], order='F')  # bellekte sütun sütun dizilir: 1,4,2,5,3,6

print("C-order flatten:", c_array.flatten(order='A'))  # 'A' mevcut bellek sırasına göre düzleştirir
print("F-order flatten:", f_array.flatten(order='A'))
print("C-order flags C_CONTIGUOUS:", c_array.flags['C_CONTIGUOUS'])
print("F-order flags F_CONTIGUOUS:", f_array.flags['F_CONTIGUOUS'])

C-order flatten: [1 2 3 4 5 6]
F-order flatten: [1 4 2 5 3 6]
C-order flags C_CONTIGUOUS: True
F-order flags F_CONTIGUOUS: True


## 2. Gelişmiş Broadcasting Kuralları ve `np.newaxis`

Broadcasting kuralı özetle şöyledir: NumPy iki array'in şekillerini **sondan başlayarak** karşılaştırır.
Boyutlar eşitse ya da biri **1** ise uyumludur. `np.newaxis`, eksik boyutu elle eklememizi sağlar.

In [5]:
a = np.array([1, 2, 3])          # şekli: (3,)
b = np.array([[10], [20], [30]])  # şekli: (3, 1)

# Broadcasting: (3,) ile (3,1) karşılaştırılır -> sonuçta (3,3) boyutunda bir array oluşur
sonuc = a + b
print("Sonuç şekli:\n", sonuc)
# a her satıra, b her sütuna "yayılarak" (broadcast) toplanır

Sonuç şekli:
 [[11 12 13]
 [21 22 23]
 [31 32 33]]


In [6]:
# np.newaxis ile bir vektörü satır ya da sütun vektörüne dönüştürebiliriz
v = np.array([1, 2, 3])            # şekli: (3,)

satir_vektor = v[np.newaxis, :]    # şekli: (1, 3) yapar -> satır vektörü
sutun_vektor = v[:, np.newaxis]    # şekli: (3, 1) yapar -> sütun vektörü

print("Satır vektör şekli:", satir_vektor.shape)
print("Sütun vektör şekli:", sutun_vektor.shape)

# Pratik kullanım: iki vektör arasında tüm ikili farkları (mesafe matrisi) tek satırda hesaplama
a = np.array([1, 5, 9])
b = np.array([2, 6])
fark_matrisi = a[:, np.newaxis] - b[np.newaxis, :]   # (3,1) - (1,2) -> (3,2) broadcasting
print("Fark matrisi:\n", fark_matrisi)

Satır vektör şekli: (1, 3)
Sütun vektör şekli: (3, 1)
Fark matrisi:
 [[-1 -5]
 [ 3 -1]
 [ 7  3]]


## 3. Sliding Window (Kayan Pencere) İşlemleri

Zaman serisi analizinde (hareketli ortalama gibi) sıkça ihtiyaç duyulan kayan pencere işlemini,
döngü kullanmadan `np.lib.stride_tricks.sliding_window_view` ile yapabiliriz.

In [7]:
from numpy.lib.stride_tricks import sliding_window_view

seri = np.array([1, 3, 5, 7, 9, 11, 13, 15])   # örnek zaman serisi

pencereler = sliding_window_view(seri, window_shape=3)   # 3 elemanlık kayan pencereler oluşturur
print("Pencereler:\n", pencereler)
# Her satır, bir önceki satırdan 1 kaydırılmış 3'lü bir gruptur

hareketli_ortalama = pencereler.mean(axis=1)   # her pencerenin ortalamasını alarak hareketli ortalamayı hesaplar
print("Hareketli ortalama (window=3):", hareketli_ortalama)

Pencereler:
 [[ 1  3  5]
 [ 3  5  7]
 [ 5  7  9]
 [ 7  9 11]
 [ 9 11 13]
 [11 13 15]]
Hareketli ortalama (window=3): [ 3.  5.  7.  9. 11. 13.]


## 4. `einsum` ile Genel Tensör İşlemleri

`np.einsum` (Einstein toplama notasyonu), matris çarpımı, iz (trace), nokta çarpım gibi birçok
doğrusal cebir işlemini **tek, esnek bir sözdizimiyle** ifade etmemizi sağlar.

In [8]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# 'ij,jk->ik' notasyonu: A'nın j eksenini B'nin j ekseniyle çarpıp toplar (klasik matris çarpımı)
matris_carpimi = np.einsum('ij,jk->ik', A, B)
print("einsum matris çarpımı:\n", matris_carpimi)
print("Doğrulama (A @ B)   :\n", A @ B)   # aynı sonucu vermeli

einsum matris çarpımı:
 [[19 22]
 [43 50]]
Doğrulama (A @ B)   :
 [[19 22]
 [43 50]]


In [9]:
A = np.array([[1, 2], [3, 4]])

iz = np.einsum('ii->', A)          # 'ii->' diagonal elemanları toplar (matrisin izi/trace)
print("İz (trace):", iz)           # 1 + 4 = 5
print("Doğrulama :", np.trace(A))

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
nokta_carpim = np.einsum('i,i->', a, b)   # aynı indeksi paylaşan elemanları çarpıp toplar (dot product)
print("Nokta çarpım:", nokta_carpim)
print("Doğrulama   :", np.dot(a, b))

İz (trace): 5
Doğrulama : 5
Nokta çarpım: 32
Doğrulama   : 32


## 5. Maskeli Array'ler (`np.ma`)

`np.ma` modülü, eksik/geçersiz verileri array'den **silmeden**, hesaplamalardan hariç tutarak
çalışmamızı sağlar. Bu, `np.nan` kullanmaktan daha esnektir çünkü her veri tipiyle çalışır.

In [10]:
veri = np.array([10, -1, 15, -1, 22, 18, -1])   # -1 değeri geçersiz ölçümü temsil ediyor

maskeli_veri = np.ma.masked_equal(veri, -1)   # -1'e eşit olan elemanları "maskele" (hesaplama dışı bırak)
print("Maskeli array:", maskeli_veri)         # maskelenen değerler "--" olarak gösterilir

print("Maskeli ortalama:", maskeli_veri.mean())   # -1 değerleri hesaba katılmadan ortalama alınır
print("Normal ortalama (yanlış sonuç):", veri.mean())  # -1'leri de içerdiği için yanıltıcıdır

print("Maske dizisi:", maskeli_veri.mask)   # hangi elemanların maskelendiğini gösteren boolean dizi

Maskeli array: [10 -- 15 -- 22 18 --]
Maskeli ortalama: 16.25
Normal ortalama (yanlış sonuç): 8.857142857142858
Maske dizisi: [False  True False  True False False  True]


## 6. Modern Rastgelelik: `Generator` API

`np.random.seed()` eski (legacy) yöntemdir. Modern NumPy'da önerilen yöntem, bağımsız ve
daha kaliteli rastgelelik sağlayan `np.random.default_rng()` fonksiyonudur.

In [11]:
rng = np.random.default_rng(seed=42)   # bağımsız bir rastgele sayı üreteci (Generator) oluşturuyoruz

print("Rastgele ondalık sayılar :", rng.random(3))          # 0-1 arasında 3 rastgele sayı
print("Rastgele tam sayılar     :", rng.integers(1, 100, size=5))  # 1-99 arasında 5 tam sayı
print("Normal dağılımdan örnek  :", rng.normal(loc=0, scale=1, size=4))  # ortalama 0, std 1

# choice: bir diziden, belirtilen olasılıklarla rastgele örnekleme yapar
secenekler = np.array(["yazı", "tura"])
atislar = rng.choice(secenekler, size=10, p=[0.5, 0.5])   # eşit olasılıkla 10 kere yazı/tura
print("10 atış sonucu           :", atislar)

# Farklı bir rng nesnesi bağımsız çalışır; global state paylaşmaz -> paralel/tekrarlanabilir kod için idealdir
rng2 = np.random.default_rng(seed=42)
print("Aynı seed aynı sonucu üretir:", np.array_equal(rng.random(0), rng2.random(0)))

Rastgele ondalık sayılar : [0.77395605 0.43887844 0.85859792]
Rastgele tam sayılar     : [ 9 70 20 10 53]
Normal dağılımdan örnek  : [ 0.1278404  -0.31624259 -0.01680116 -0.85304393]
10 atış sonucu           : ['yazı' 'tura' 'tura' 'tura' 'yazı' 'yazı' 'tura' 'yazı' 'tura' 'tura']
Aynı seed aynı sonucu üretir: True


## 7. Hızlı Fourier Dönüşümü (FFT)

FFT, bir zaman serisindeki gizli **frekans bileşenlerini** ortaya çıkarır. Sinyal işleme,
ses analizi ve titreşim analizinde yaygın olarak kullanılır.

In [12]:
# İki farklı frekansta sinüs dalgasının toplamından oluşan sentetik bir sinyal üretelim
orneklem_hizi = 500                       # saniyede kaç örnek alındığı (Hz)
zaman = np.linspace(0, 1, orneklem_hizi, endpoint=False)   # 1 saniyelik zaman ekseni

sinyal = np.sin(2 * np.pi * 5 * zaman) + 0.5 * np.sin(2 * np.pi * 50 * zaman)
# sinyal, 5 Hz'lik ve 50 Hz'lik iki bileşenin toplamı

fft_sonucu = np.fft.fft(sinyal)            # sinyali frekans uzayına dönüştürür
frekanslar = np.fft.fftfreq(len(sinyal), d=1/orneklem_hizi)  # her FFT bileşeninin karşılık geldiği frekans

genlikler = np.abs(fft_sonucu)             # karmaşık sayıların büyüklüğünü (genliği) alırız

# Sadece pozitif frekansları ve en baskın 2 frekansı bulalım
pozitif_maske = frekanslar > 0
en_guclu_indeksler = np.argsort(genlikler[pozitif_maske])[-2:]   # en yüksek genlikli 2 frekans
print("Tespit edilen baskın frekanslar (Hz):", np.sort(frekanslar[pozitif_maske][en_guclu_indeksler]))
# Beklenen sonuç: yaklaşık [5. 50.]

Tespit edilen baskın frekanslar (Hz): [ 5. 50.]


## 8. Polinom Uydurma (Curve Fitting)

Elimizdeki noktalara en iyi uyan polinom eğrisini bulmak için `np.polyfit` kullanılır.
Bu, basit regresyon/eğri uydurma problemleri için hızlı bir çözümdür.

In [13]:
x = np.array([0, 1, 2, 3, 4, 5])
y = np.array([1.1, 3.9, 9.2, 15.8, 25.1, 36.2])   # yaklaşık y = x^2 + 2x + 1 şeklinde gürültülü veri

katsayilar = np.polyfit(x, y, deg=2)   # veriye en iyi uyan 2. dereceden (ax^2 + bx + c) polinomu bulur
print("Bulunan katsayılar (a, b, c):", katsayilar)

polinom = np.poly1d(katsayilar)   # katsayılardan çağrılabilir bir polinom fonksiyonu oluşturur
print("x=6 için tahmin edilen y  :", polinom(6))   # modeli yeni bir nokta için kullanıyoruz

# Modelin orijinal noktalara ne kadar iyi uyduğunu karşılaştıralım
print("Gerçek y değerleri  :", y)
print("Modelin tahminleri  :", polinom(x))

Bulunan katsayılar (a, b, c): [1.02678571 1.88607143 1.08928571]
x=6 için tahmin edilen y  : 49.37000000000001
Gerçek y değerleri  : [ 1.1  3.9  9.2 15.8 25.1 36.2]
Modelin tahminleri  : [ 1.08928571  4.00214286  8.96857143 15.98857143 25.06214286 36.18928571]


## 9. Tarih/Zaman Verisiyle Çalışma (`datetime64`, `timedelta64`)

NumPy, tarih ve zaman verilerini verimli şekilde saklamak ve üzerinde aritmetik işlem yapmak için
özel `datetime64` ve `timedelta64` tiplerini sunar.

In [14]:
tarih = np.array('2024-01-15', dtype='datetime64[D]')   # gün (Day) hassasiyetinde bir tarih
print("Tarih:", tarih)

# arange ile bir tarih aralığı (gün gün) üretebiliriz
tarih_araligi = np.arange('2024-01-01', '2024-01-10', dtype='datetime64[D]')
print("Tarih aralığı:", tarih_araligi)

# İki tarih arasındaki farkı alırsak otomatik olarak timedelta64 elde ederiz
fark = np.datetime64('2024-06-01') - np.datetime64('2024-01-01')
print("İki tarih arasındaki gün farkı:", fark)

# Bir tarihe gün ekleyebiliriz
yeni_tarih = tarih + np.timedelta64(30, 'D')   # tarihe 30 gün ekler
print("30 gün sonrası:", yeni_tarih)

Tarih: 2024-01-15
Tarih aralığı: ['2024-01-01' '2024-01-02' '2024-01-03' '2024-01-04' '2024-01-05'
 '2024-01-06' '2024-01-07' '2024-01-08' '2024-01-09']
İki tarih arasındaki gün farkı: 152 days
30 gün sonrası: 2024-02-14


## 10. Kısmi Sıralama: `argpartition` ile Performans Kazanımı

Elimizdeki dizinin **tamamını** sıralamak yerine sadece "en büyük/küçük k eleman" gerekiyorsa,
`np.argpartition` tam sıralamadan çok daha hızlı çalışır (O(n) civarı, sort ise O(n log n)).

In [15]:
buyuk_dizi = np.random.randint(0, 1_000_000, size=1_000_000)   # 1 milyon elemanlı rastgele dizi

k = 5   # en küçük 5 elemanı bulmak istiyoruz

# Yöntem 1: tam sıralama (yavaş, çünkü tüm diziyi sıralar)
import time
baslangic = time.time()
tam_siralama_sonucu = np.sort(buyuk_dizi)[:k]
sort_suresi = time.time() - baslangic

# Yöntem 2: argpartition (hızlı, çünkü sadece ilk k elemanı doğru konuma yerleştirir, geri kalanı sıralamaz)
baslangic = time.time()
kismi_indeksler = np.argpartition(buyuk_dizi, k)[:k]   # en küçük k elemanın indekslerini (sırasız) verir
kismi_sonuc = np.sort(buyuk_dizi[kismi_indeksler])     # sadece bu 5 elemanı sıralıyoruz (çok ucuz)
partition_suresi = time.time() - baslangic

print("Tam sıralama sonucu :", tam_siralama_sonucu)
print("argpartition sonucu :", kismi_sonuc)
print(f"sort süresi        : {sort_suresi:.5f} sn")
print(f"argpartition süresi: {partition_suresi:.5f} sn")

Tam sıralama sonucu : [0 0 1 2 3]
argpartition sonucu : [0 0 1 2 3]
sort süresi        : 0.04496 sn
argpartition süresi: 0.03984 sn


## 11. `apply_along_axis` ile Eksen Bazlı Özel Fonksiyonlar

Hazır bir NumPy fonksiyonu olmayan özel bir işlemi, bir array'in belirli bir ekseni boyunca
uygulamak istediğimizde `np.apply_along_axis` kullanılır.

In [16]:
def aralik_genisligi(satir):
    # kendi tanımladığımız fonksiyon: bir dizinin maksimum ve minimum değeri arasındaki farkı döndürür
    return satir.max() - satir.min()

m = np.array([[1, 5, 3],
              [10, 2, 8],
              [4, 4, 4]])

# axis=1 -> fonksiyonu HER SATIRA uygular (satır boyunca ilerler)
satir_bazli = np.apply_along_axis(aralik_genisligi, axis=1, arr=m)
print("Her satırın aralık genişliği:", satir_bazli)   # [4 8 0]

# axis=0 -> fonksiyonu HER SÜTUNA uygular (sütun boyunca ilerler)
sutun_bazli = np.apply_along_axis(aralik_genisligi, axis=0, arr=m)
print("Her sütunun aralık genişliği:", sutun_bazli)   # [9 3 5]

Her satırın aralık genişliği: [4 8 0]
Her sütunun aralık genişliği: [9 3 5]


## 12. `memmap` ile Bellek Sığmayan Büyük Dosyalarla Çalışma

`np.memmap`, disk üzerindeki bir dosyayı **tamamen belleğe yüklemeden** array gibi kullanmamızı
sağlar. Bellek boyutunu aşan devasa veri setleriyle çalışırken hayat kurtarır.

In [17]:
# Örnek olarak diskte 1000x1000 boyutunda float32 bir memmap dosyası oluşturuyoruz
sekil = (1000, 1000)
mm = np.memmap('buyuk_veri.dat', dtype='float32', mode='w+', shape=sekil)
# mode='w+' -> yeni dosya oluştur ve yaz; veri RAM'e değil doğrudan diske yazılır

mm[:] = np.random.rand(*sekil)   # rastgele verilerle dolduruyoruz (arka planda parça parça diske yazılır)
mm.flush()   # değişikliklerin diske kesin olarak yazıldığından emin oluyoruz

# Dosyayı tekrar, sadece OKUMA modunda ve belleğe tam yüklemeden açıyoruz
mm_okuma = np.memmap('buyuk_veri.dat', dtype='float32', mode='r', shape=sekil)

print("Şekil          :", mm_okuma.shape)
print("İlk 3x3 alt blok:\n", mm_okuma[:3, :3])   # sadece ihtiyaç duyduğumuz kısmı diskten okur
print("Tüm verinin ortalaması:", mm_okuma.mean())  # normal array gibi hesaplama yapılabilir

del mm, mm_okuma   # dosya tanıtıcılarını serbest bırakıyoruz
import os
os.remove('buyuk_veri.dat')   # örnek dosyayı temizliyoruz

Şekil          : (1000, 1000)
İlk 3x3 alt blok:
 [[0.8657464  0.3689364  0.50397426]
 [0.92649186 0.31988052 0.31910178]
 [0.68842894 0.6163185  0.33183917]]
Tüm verinin ortalaması: 0.50022465


## Özet

Bu ileri düzey notebook'ta öğrendiklerimiz:

- **strides** ve bellek düzeni (C-order / F-order) ile transpose'un neden "ücretsiz" olduğu
- `np.newaxis` ve gelişmiş broadcasting ile ikili mesafe/fark matrisleri hesaplama
- `sliding_window_view` ile döngüsüz hareketli pencere işlemleri
- `einsum` ile matris çarpımı, iz ve nokta çarpımın genel notasyonu
- `np.ma` ile eksik/geçersiz verileri silmeden yönetme
- Modern `Generator` (`default_rng`) API'si ile bağımsız rastgelelik
- `np.fft` ile sinyallerin frekans bileşenlerini bulma
- `np.polyfit` / `np.poly1d` ile eğri uydurma
- `datetime64` / `timedelta64` ile tarih aritmetiği
- `argpartition` ile tam sıralamadan çok daha hızlı "en büyük/küçük k eleman" bulma
- `apply_along_axis` ile eksen bazlı özel fonksiyon uygulama
- `memmap` ile RAM'e sığmayan devasa dosyalarla çalışma

Buradan sonrası için: NumPy'ın C-API'si ile hızlandırma, `numba`/`cython` ile JIT derleme,
GPU üzerinde NumPy benzeri işlemler için `cupy`, ve büyük ölçekli paralel hesaplama için `dask.array`.
